# 📘 Agentic Architectures 3 (Agno): ReAct (Reason + Act)

This notebook is the **Agno-framework** counterpart of `03_ReAct.ipynb`. The original notebook tells a two-part story: first a *basic* single-shot tool-using agent that fails at a multi-hop question, then a *ReAct* agent that succeeds by looping `think → act → observe` until enough evidence is gathered.

In LangGraph the difference between the two is literally **one edge**: the basic graph has `tools → END`, the ReAct graph has `tools → agent` (a back-edge that creates the loop). That observation is the headline of `docs/03_ReAct_analysis.md` §3.

In Agno that edge is **internal to `agent.run()`** — there is no graph to wire. To replicate the basic-vs-ReAct contrast we instead manipulate `tool_call_limit`: setting it to `1` produces single-shot behaviour, leaving it at a higher value (or letting the model decide) reproduces multi-hop ReAct.

### Mapping the abstraction

| Concern | LangGraph (`03_ReAct.ipynb`) | Agno (this notebook) |
|---|---|---|
| Basic single-shot agent | `StateGraph` with `agent → tools → END` (no back-edge) | `Agent(..., tool_call_limit=1)` |
| ReAct loop | `StateGraph` with `agent → tools → agent` (back-edge) | `Agent(..., tool_call_limit=N)` — loop is internal |
| Decision logic | `tools_condition` reads `last_message.tool_calls` | Hidden inside `agent.run()` |
| Driving execution | `app.stream(initial_input, stream_mode='values')` | `agent.run(query)` |
| Trace inspection | Iterate stream chunks, `pretty_print` each | Iterate `response.messages` after the run |

The user-facing behaviour is the same: the basic agent fails the multi-hop query, the ReAct agent succeeds.

## Phase 0: Foundation & Setup

In [ ]:
# !pip install -q -U agno openai python-dotenv rich pydantic

### Step 0.2: Importing Libraries and Setting Up Keys

**Action Required:** Create a `.env` file in this directory with:
```
SILICONFLOW_API_KEY="sk-..."
TAVILY_API_KEY="your_tavily_api_key_here"
```

In [ ]:
import os
from typing import Any
from dotenv import load_dotenv

from pydantic import BaseModel, Field

from agno.agent import Agent
from agno.models.openai import OpenAILike
from agno.tools.tavily import TavilyTools

from rich.console import Console
from rich.markdown import Markdown

load_dotenv()

for key in ["SILICONFLOW_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        print(f"{key} not found. Please create a .env file and set it.")

console = Console()
print("Environment variables loaded.")

## Phase 1: Shared Model and Tool

Both the basic and the ReAct agents share the same LLM and search tool. The only structural difference will be the `tool_call_limit` we hand each one.

In [ ]:
model = OpenAILike(
    id="deepseek-ai/DeepSeek-V3",
    api_key=os.environ.get("SILICONFLOW_API_KEY"),
    base_url="https://api.siliconflow.cn/v1",
    temperature=0,
)

search_tool = TavilyTools(
    api_key=os.environ.get("TAVILY_API_KEY"),
    search_depth="advanced",
    format="markdown",
    include_answer=True,
)

print(f"Model: {model.id}")
print(f"Tool: {type(search_tool).__name__}")

## Phase 2: The Basic Approach — Single-Shot Tool User (`tool_call_limit=1`)

We instantiate an Agno agent and **cap it at a single tool call**. After the tool returns, the agent must produce a final answer without being able to look anything else up. This is the Agno-equivalent of LangGraph's `tools → END` edge.

In [ ]:
basic_agent = Agent(
    name="Basic Tool User",
    model=model,
    tools=[search_tool],
    instructions=[
        "You are a helpful assistant with access to a web search tool.",
        "You may call the tool at most once. After the tool returns, you must produce a final answer.",
    ],
    add_datetime_to_context=True,
    markdown=True,
    telemetry=False,
    tool_call_limit=1,  # this single flag is the difference between basic and ReAct in Agno
)

print("Basic single-shot Agno agent built.")

### Step 2.2: Testing the Basic Agent on a Multi-Step Problem

Same multi-hop query as the original notebook. With a single tool call budget, the agent typically returns either an incomplete answer or admits it could not find part of the information.

In [ ]:
multi_step_query = (
    "Who is the current CEO of the company that created the sci-fi movie 'Dune', "
    "and what was the budget for that company's most recent film?"
)

console.print(f"[bold yellow]Testing BASIC agent (tool_call_limit=1):[/bold yellow] '{multi_step_query}'\n")

basic_response = basic_agent.run(multi_step_query)

console.print("--- [bold red]Final Output from Basic Agent[/bold red] ---")
console.print(Markdown(basic_response.content or "(no content)"))

tool_call_count_basic = sum(
    len(getattr(m, 'tool_calls', None) or []) for m in (basic_response.messages or [])
)
console.print(f"\n[dim]Total tool calls: {tool_call_count_basic}[/dim]")

**Discussion:** Bounded to a single tool call, the agent must compress the entire multi-hop question into one search query. Tavily returns broad, generic results, and the agent has no way to chase the follow-up facts (CEO name, latest film, budget). The failure mode is exactly the same as the LangGraph basic agent's — the architectural deficiency is identical because the *control flow* is identical: one think, one act, one observe, stop.

## Phase 3: The Advanced Approach — ReAct (`tool_call_limit=6`)

We rebuild the agent with a higher tool call budget so it can loop through `think → act → observe` until it has all the pieces. Crucially we also tighten the instructions to *encourage* iterative decomposition — without this hint, some models will still try to one-shot the query even with a higher cap.

In [ ]:
react_agent = Agent(
    name="ReAct Agent",
    model=model,
    tools=[search_tool],
    instructions=[
        "You are a research assistant with access to a web search tool.",
        "For multi-hop questions, decompose the question into sub-questions and search for each one in turn.",
        "After each search, reason about whether you have enough information to answer, or whether another search is needed.",
        "When you have all the pieces, synthesise a single concise final answer.",
    ],
    add_datetime_to_context=True,
    markdown=True,
    telemetry=False,
    tool_call_limit=6,  # ReAct loop budget — enough for typical 2-4 hop questions
)

print("ReAct Agno agent built with iterative tool-call budget.")

In [ ]:
console.print(f"[bold green]Testing ReAct agent on the same multi-step query:[/bold green] '{multi_step_query}'\n")

react_response = react_agent.run(multi_step_query)

console.print("--- [bold green]Final Output from ReAct Agent[/bold green] ---")
console.print(Markdown(react_response.content or "(no content)"))

tool_call_count_react = sum(
    len(getattr(m, 'tool_calls', None) or []) for m in (react_response.messages or [])
)
console.print(f"\n[dim]Total tool calls: {tool_call_count_react}[/dim]")

### Step 3.2: Inspecting the Reasoning Loop

To make the loop visible — which `app.stream(..., stream_mode='values')` did for free in the LangGraph notebook — we walk `response.messages` after the fact and print each role + tool call. This is the Agno equivalent of LangGraph's per-chunk `pretty_print`.

In [ ]:
def summarise_messages(messages: list[Any]) -> None:
    for i, m in enumerate(messages or []):
        role = getattr(m, 'role', '?')
        tool_calls = getattr(m, 'tool_calls', None) or []
        content = getattr(m, 'content', None) or ''
        snippet = (content[:200] + '…') if isinstance(content, str) and len(content) > 200 else content
        console.print(f"[bold]{i:02d}[/bold] [yellow]{role}[/yellow] tool_calls={len(tool_calls)}")
        for tc in tool_calls:
            fn = tc.get('function', {}) if isinstance(tc, dict) else {}
            console.print(f"     [cyan]→ {fn.get('name')} args={fn.get('arguments')}[/cyan]")
        if snippet:
            console.print(f"     {snippet}")

console.print("[bold]--- ReAct Execution Trace ---[/bold]")
summarise_messages(react_response.messages)

**Discussion:** Reading the trace top-to-bottom you should see the classic ReAct rhythm: assistant message with `tool_calls`, then a `tool` message with the search result, then another assistant message with the next `tool_calls`, and so on, until a final assistant message with no tool calls — the synthesised answer.

Compared to LangGraph this trace is *post-hoc* (you read it from `response.messages` after the run completes) rather than *streamed* (chunks yielded as they happen). For analysis they carry the same information; for live UIs the LangGraph variant has a slight ergonomic edge, while Agno offers `agent.run(..., stream=True, stream_events=True)` to get a stream of typed events if you need it.

## Phase 4: Quantitative Evaluation

Same LLM-as-a-Judge setup as the original notebook, using Agno's native structured output (`output_schema=...`, `use_json_mode=True`).

In [ ]:
class TaskEvaluation(BaseModel):
    task_completion_score: int = Field(description="Score 1-10 on whether the agent successfully completed all parts of the user's request.")
    reasoning_quality_score: int = Field(description="Score 1-10 on the logical flow and reasoning process demonstrated by the agent.")
    justification: str = Field(description="A brief justification for the scores.")

judge = Agent(
    name="Judge",
    model=model,
    instructions=[
        "You are an expert judge of AI agents.",
        "Score the trace on a scale of 1-10 for each criterion.",
        "A 10 means the task was completed perfectly; a 1 means complete failure.",
    ],
    output_schema=TaskEvaluation,
    use_json_mode=True,
    telemetry=False,
)

def trace_text(response) -> str:
    return "\n".join(
        f"{getattr(m, 'role', '?')}: {getattr(m, 'content', '') or ''} tool_calls={getattr(m, 'tool_calls', None) or ''}"
        for m in (response.messages or [])
    )

def evaluate(query: str, response) -> TaskEvaluation:
    prompt = (
        f"User's task:\n{query}\n\n"
        f"Full agent conversation trace:\n```\n{trace_text(response)}\n```"
    )
    return judge.run(prompt).content

console.print("--- Evaluating Basic Agent's Output ---")
basic_eval = evaluate(multi_step_query, basic_response)
console.print(basic_eval)

console.print("\n--- Evaluating ReAct Agent's Output ---")
react_eval = evaluate(multi_step_query, react_response)
console.print(react_eval)

## Conclusion

We re-implemented the **ReAct** architecture in Agno and reproduced the original notebook's headline contrast: a single-shot agent fails the multi-hop query, while a looping ReAct agent succeeds. The translation reveals an interesting structural point:

- In **LangGraph**, the difference between the two architectures is a single edge in the state graph (`tools → END` vs `tools → agent`).
- In **Agno**, the difference is a single integer (`tool_call_limit=1` vs a higher value).

Both expose ReAct as a *one-knob change* from basic tool use — they just expose the knob at different layers (graph topology vs runtime budget). For benchmark purposes this means the LangGraph and Agno ReAct implementations are directly comparable: same tool, same LLM, same query, same expected behaviour. The only thing that differs is the orchestrator's overhead.